In [ ]:
# Force install the stable version of datasets that allows remote scripts
!pip install "datasets==2.21.0" transformers seqeval evaluate accelerate -U -q

# --- Task 1: Dataset Selection ---[cite: 3]
from datasets import load_dataset

print("Downloading the CoNLL-2003 Dataset...")
# Now that we are on the stable version, trust_remote_code will work perfectly!
dataset = load_dataset("eriktks/conll2003", trust_remote_code=True)

# CoNLL-2003 contains POS tags, Chunk tags, and NER tags. We will focus on Chunk tags[cite: 3].
chunk_tags = dataset["train"].features["chunk_tags"].feature.names

print(f"\nTask 1 Deliverables: Dataset Name: CoNLL-2003")
print(f"Label Categories (Chunk Tags): {chunk_tags}")

# --- Task 3 Requirement: Proper label mapping (id2label, label2id) ---[cite: 3]
id2label = {i: label for i, label in enumerate(chunk_tags)}
label2id = {label: i for i, label in enumerate(chunk_tags)}

print(f"\nDataset loaded successfully! We have {len(dataset['train'])} training sentences.")

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]


Task 1 Deliverables: Dataset Name: CoNLL-2003
Label Categories (Chunk Tags): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']

Dataset loaded successfully! We have 14041 training sentences.


In [ ]:
from transformers import AutoTokenizer

# Load the BERT tokenizer as required by Task 2[cite: 3]
print("Loading the BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# --- Let's look at the downloaded data! ---
print("\n--- Here is sentence #1 inside your dataset variable ---")
print("Words:", dataset["train"][0]["tokens"])
print("Tags (Numerical):", dataset["train"][0]["chunk_tags"])

# --- Task 2: Data Preprocessing (Tokenization & Label Alignment) ---[cite: 3]
def tokenize_and_align_labels(examples):
    # Tokenize the words[cite: 3]
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens (like [CLS] and [SEP]) get a -100 label so the model ignores them[cite: 3]
            if word_idx is None:
                label_ids.append(-100)
            # Only label the very first piece of a given word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Handle subwords: if a word is split, label the subwords -100[cite: 3]
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("\nAligning labels and handling subwords with -100...")
# Apply this function to our entire dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

print("Task 2 Preprocessing Complete! We now have input_ids, attention_mask, and labels ready.")


Loading the BERT tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


--- Here is sentence #1 inside your dataset variable ---
Words: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Tags (Numerical): [11, 21, 11, 12, 21, 22, 11, 12, 0]

Aligning labels and handling subwords with -100...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Task 2 Preprocessing Complete! We now have input_ids, attention_mask, and labels ready.


In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import torch

# --- Task 3: Model Setup (15%) ---[cite: 1]
print("Loading the BERT model for Token Classification...")

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(chunk_tags),
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded and moved to: {device}")


# --- Task 4: Training (20%) ---[cite: 1]
print("\nSetting up Training Arguments...")

# The Data Collator automatically pads our inputs/labels dynamically during training
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Define our hyperparameters
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
)

# Use the Hugging Face Trainer to handle the training loop[cite: 1]
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(1000)),
    eval_dataset=tokenized_datasets["validation"].select(range(200)),
    processing_class=tokenizer,  # <--- FIXED: 'tokenizer' is now 'processing_class'!
    data_collator=data_collator,
)

print("\nStarting Training! Grab a coffee, this will take a few minutes...")
# Train the model on the selected dataset[cite: 1]
trainer.train()
print("\nTraining Complete!")

Loading the BERT model for Token Classification...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

Model loaded and moved to: cpu

Setting up Training Arguments...

Starting Training! Grab a coffee, this will take a few minutes...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.236103,1.162487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Training Complete!


In [1]:
# --- Setup: Install everything cleanly ---
!pip install "datasets==2.21.0" transformers evaluate seqeval accelerate -U -q

import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline

print("--- Task 1: Dataset Selection ---")
# Download the dataset and extract the labels[cite: 1]
dataset = load_dataset("eriktks/conll2003", trust_remote_code=True)
chunk_tags = dataset["train"].features["chunk_tags"].feature.names
id2label = {i: label for i, label in enumerate(chunk_tags)}
label2id = {label: i for i, label in enumerate(chunk_tags)}

print("--- Task 2: Data Preprocessing ---")
# Load tokenizer and align subword labels with -100[cite: 1]
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

print("--- Task 3 & 4: Model Setup & Training ---")
# Setup BERT and train for 1 epoch on a small subset[cite: 1]
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased", num_labels=len(chunk_tags), id2label=id2label, label2id=label2id
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir="./results", eval_strategy="epoch", learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=1, weight_decay=0.01, logging_steps=10,
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(1000)),
    eval_dataset=tokenized_datasets["validation"].select(range(200)),
    processing_class=tokenizer, data_collator=data_collator,
)
trainer.train()

print("\n--- Task 5: Evaluation Metrics ---")
# Calculate seqeval metrics without the -100 tokens[cite: 1]
metric = evaluate.load("seqeval")
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"].select(range(200)))
predictions = np.argmax(predictions, axis=2)

true_predictions = [[chunk_tags[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
true_labels = [[chunk_tags[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
results = metric.compute(predictions=true_predictions, references=true_labels)

print(f"Precision: {results['overall_precision']:.4f}")
print(f"Recall:    {results['overall_recall']:.4f}")
print(f"F1 Score:  {results['overall_f1']:.4f}")

print("\n--- Task 6: Inference ---")
# Predict chunk tags on a custom sentence[cite: 1]
chunker = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple", device=device)
custom_sentence = "John works at Google in California."
outputs = chunker(custom_sentence)
print(f"Input: {custom_sentence}")
for entity in outputs:
    print(f"Word: {entity['word']} | Tag: {entity['entity_group']} | Confidence: {entity['score']:.4f}")

--- Task 1: Dataset Selection ---


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

--- Task 2: Data Preprocessing ---


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Task 5: Evaluation Metrics ---


Precision: 0.4775
Recall:    0.4426
F1 Score:  0.4594

--- Task 6: Inference ---
Input: John works at Google in California.
Word: john works | Tag: NP | Confidence: 0.4184
Word: at | Tag: NP | Confidence: 0.2895
Word: google | Tag: NP | Confidence: 0.4546
Word: in california | Tag: NP | Confidence: 0.3695


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


### Task 7: Comparison (10%)[cite: 1]
* **POS Tagging (Grammar-level):** This is considered the "easier" task as it assigns a single grammatical label (Noun, Verb, Adjective) to individual words independent of the broader phrase context.
* **Chunking (Phrase-level):** This is a "medium" difficulty task because the model must group multiple words together into structural phrases (like Noun Phrases or Verb Phrases) requiring a deeper contextual understanding of the sentence boundaries.

### Task 8: Report / Blog (5%)[cite: 1]
* **Differences:** While POS tagging identifies what a word *is*, Chunking identifies how words *work together* as a unit within a sentence.
* **Challenges Faced:** The most significant challenge was navigating Hugging Face library versioning updates. The `datasets` library v4.0.0 deprecated `trust_remote_code` and blocked execution scripts, requiring a manual downgrade to v2.21.0 to successfully load the CoNLL-2003 dataset. Additionally, the `Trainer` class updated the `tokenizer` argument to `processing_class`, requiring syntax adaptation.
* **Observations & Insights:** Successfully managing subword tokenization required aligning labels with the `-100` special token to prevent the model from calculating loss on fragmented words. Even with a short 1-epoch training cycle on a CPU, the BERT architecture quickly adapted to recognizing foundational chunking patterns.